In [0]:
# %sql
# SELECT
#   A.PRODUCT_ID,
#   A.available_to_sell AS AVAILABLE_INVENTORY,
#   A.RESERVED_QUANTITY AS RESERVED_INVENTORY,
#   A.INVENTORY_TYPE AS INVENTORY_STATUS,
#   SUM(C.QUANTITY) AS SALES_QUANTITY
# FROM
#   ECOMMERCE.SILVER.INVENTORY A
#     INNER JOIN ECOMMERCE.SILVER.ORDER_ITEMS C
#       ON A.PRODUCT_ID = C.PRODUCT_ID
#     INNER JOIN ECOMMERCE.SILVER.ORDERS B
#       ON C.ORDER_ID = B.ORDER_ID
# WHERE
#   B.ORDER_STATUS = 'DELIVERED'
# GROUP BY
#   1,
#   2,
#   3,
#   4
# LIMIT 10;

In [0]:
from pyspark.sql import functions as F



In [0]:
inventory = spark.table("ECOMMERCE.SILVER.INVENTORY")
order_items = spark.table("ECOMMERCE.SILVER.ORDER_ITEMS")
orders = spark.table("ECOMMERCE.SILVER.ORDERS")


In [0]:
inventory_sales = (
    inventory.alias("A")
    .join(
        order_items.alias("C"), F.col("A.PRODUCT_ID") == F.col("C.PRODUCT_ID"), "inner"
    )
    .join(orders.alias("B"), F.col("C.ORDER_ID") == F.col("B.ORDER_ID"), "inner")
    .filter(F.col("B.ORDER_STATUS") == "DELIVERED")
    .groupBy(
        F.col("A.PRODUCT_ID"),
        F.col("A.available_to_sell"),
        F.col("A.RESERVED_QUANTITY"),
        F.col("A.INVENTORY_TYPE"),
    )
    .agg(F.sum("C.QUANTITY").alias("SALES_QUANTITY"))
    .select(
        F.col("A.PRODUCT_ID"),
        F.col("A.available_to_sell").alias("AVAILABLE_INVENTORY"),
        F.col("A.RESERVED_QUANTITY").alias("RESERVED_INVENTORY"),
        F.col("A.INVENTORY_TYPE").alias("INVENTORY_STATUS"),
        F.col("SALES_QUANTITY"),
    )
)

In [0]:
display(inventory_sales)

In [0]:
inventory_sales.write.format("delta").mode("APPEND").saveAsTable(
    "ecommerce.gold.inventory_sales"
)

In [0]:
high_selling_out_of_stock = (
    inventory_sales.filter(F.col("AVAILABLE_INVENTORY") == 0)
    .filter(
        F.col("SALES_QUANTITY")
        >= inventory_sales.approxQuantile("SALES_QUANTITY", [0.50], 0.01)[0]
    )
    .orderBy(F.col("SALES_QUANTITY").desc())
)

display(high_selling_out_of_stock)

In [0]:
products_need_replenishment = (
    inventory_sales.filter(F.col("AVAILABLE_INVENTORY") <= F.col("SALES_QUANTITY"))
    .filter(F.col("INVENTORY_STATUS") != "IN_STOCK")
    .orderBy(F.col("SALES_QUANTITY").desc())
)

display(products_need_replenishment)

In [0]:
excess_inventory = inventory_sales.filter(
    F.col("AVAILABLE_INVENTORY") > 3 * F.col("SALES_QUANTITY")
).orderBy(F.col("AVAILABLE_INVENTORY").desc())

display(excess_inventory)